# Module `visualization.py`

Ce notebook montre comment visualiser les graphes CESIPATH avec `matplotlib`.

Il sert a comprendre visuellement les etats du reseau : graphe de base, graphe residuel et graphe dynamique.

## Attention sur les boutons dans Jupyter

Le bouton `->` est implemente avec `matplotlib.widgets.Button`.

Selon le frontend Jupyter, ce bouton peut ne pas reagir correctement. Pour une interaction fiable, le script suivant est recommande :

```bash
python3 main_visualization.py
```

Dans ce notebook, une cellule de fallback permet aussi d'avancer manuellement d'un tour.

In [ ]:
%matplotlib notebook

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.visualization import GraphVisualizer

## Creation d'une instance a visualiser

On genere un graphe de taille modeste pour que les couts restent lisibles sur les aretes.

Les couleurs de la visualisation correspondent aux etats des routes :

- vert : route active ;
- violet : route surchargee ;
- rouge pointille : route interdite statiquement ;
- orange pointille : route temporairement indisponible dynamiquement.

In [ ]:
config = GraphGenerationConfig(
    node_count=8,
    seed=42,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    dynamic_sigma=0.18,
    dynamic_mean_reversion_strength=0.35,
    dynamic_max_multiplier=1.8,
    dynamic_forbid_probability=0.03,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)

generator = GraphGenerator(config)
instance = generator.generate()
instance.summary()

## Graphe de base

Le graphe de base montre uniquement les routes physiques generees et leurs couts de base.

A ce stade, aucune contrainte statique ou dynamique n'est appliquee.

In [ ]:
visualizer = GraphVisualizer(instance, generator)
visualizer.show_base_graph()

## Graphe residuel

Le graphe residuel applique les contraintes statiques :

- les routes interdites sont affichees en rouge pointille ;
- les routes surchargees sont en violet ;
- les routes libres restent actives.

C'est a partir de ce graphe que Dijkstra construit la matrice complete.

In [ ]:
visualizer.show_residual_graph()

## Graphe dynamique

Le graphe dynamique represente l'etat courant du reseau.

Le bouton `->` fait avancer d'un tour : les couts dynamiques changent, certaines routes peuvent devenir temporairement indisponibles, puis Dijkstra recalcule la completion.

In [ ]:
session = visualizer.show_dynamic_graph()
session.fig

## Fallback si le bouton ne marche pas

Si le bouton `->` ne reagit pas dans ton notebook, execute la cellule suivante pour avancer manuellement d'un tour.

Elle appelle la meme methode que le bouton.

In [ ]:
visualizer.advance_session(session)
session.fig